# Dual-GPU GFlowNet TB/DB → Bayesian Elite (one seed)

Yêu cầu Kaggle Accelerator: **GPU T4 x2**. Notebook dùng output các stage trước làm Add Input, pin chính xác một GitHub commit, rồi chạy TB trên GPU 0 và DB trên GPU 1. Sau Bayesian Elite, nhánh TB chạy thêm Stage 5 Bayesian chuẩn bằng frozen diverse sampler TB.

In [ ]:
DATASET_ID = "REPLACE_DATASET_ID"
DATA_DIR = "REPLACE_DATA_DIR"
SEED = 42
BACKBONE = "vit_b_32"
PRIOR_RUN_ID = "REPLACE_PRIOR_RUN_ID"
GIT_COMMIT = "REPLACE_GIT_COMMIT"
MC_SAMPLES = 32
GFLOWNET_ITERATIONS = 5000
NUM_EPOCHS = 100
PATIENCE = 5

In [ ]:
# Generator có thể thay cell này bằng symlink setup riêng của dataset.
from pathlib import Path
assert Path(DATA_DIR).exists(), f'DATA_DIR không tồn tại: {DATA_DIR}'

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/khoaddb2207532/neuro_symbolic_mlops_l2_app.git"
PROJECT = Path("/kaggle/working/neuro_symbolic_mlops_l2_app")
if not PROJECT.exists():
    clone_url = REPO_URL
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        if token:
            clone_url = REPO_URL.replace("https://", f"https://{token}@")
    except Exception:
        pass
    subprocess.run(["git", "clone", "--filter=blob:none", clone_url, str(PROJECT)], check=True)
subprocess.run(["git", "fetch", "origin", GIT_COMMIT, "--depth", "1"], cwd=PROJECT, check=True)
subprocess.run(["git", "checkout", "--detach", GIT_COMMIT], cwd=PROJECT, check=True)
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT, text=True).strip()
assert actual == GIT_COMMIT, (actual, GIT_COMMIT)
print("Pinned commit:", actual)

In [ ]:
%cd /kaggle/working/neuro_symbolic_mlops_l2_app
!pip install -q torchgfn tensordict dvclive dvc openpyxl
!nvidia-smi -L
import torch
assert torch.cuda.device_count() >= 2, 'Hãy chọn Kaggle Accelerator: GPU T4 x2'
print('CUDA devices:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
import subprocess
import sys

RUN_ROOT = f"/kaggle/working/dual_elite_seed_{SEED}"
command = [
    sys.executable, '-m', 'pipelines.run_dual_gpu_elite_seed_experiment',
    '--config', 'params.yaml',
    '--seed', str(SEED),
    '--dataset-id', DATASET_ID,
    '--prior-run-id', PRIOR_RUN_ID,
    '--backbone', BACKBONE,
    '--data-dir', DATA_DIR,
    '--output-dir', RUN_ROOT,
    '--project-dir', '/kaggle/working/neuro_symbolic_mlops_l2_app',
    '--kaggle-input-root', '/kaggle/input',
    '--mc-samples', str(MC_SAMPLES),
    '--gflownet-iterations', str(GFLOWNET_ITERATIONS),
    '--num-epochs', str(NUM_EPOCHS),
    '--patience', str(PATIENCE),
]
print(' '.join(command), flush=True)
subprocess.run(command, check=True)

## Resume

Nếu notebook bị ngắt, Save Version để giữ output rồi Add Input version đó vào lần chạy sau. Runner nhận diện `dual_elite_manifest.json`, phục hồi hai nhánh và tiếp tục Stage 5 từ `training_last.pth`.